# SatQuery AI — Division 2: Single-Image Remote-Sensing Intelligence
## Google Colab GPU Compute Pipeline & Scientific Training Runner

- **Division**: Division 2 (Single-Image Remote-Sensing Intelligence: VQA + Visual Grounding)
- **Lead Owner**: Sruthi (`sruthi-270` / `rajamanurisruthi@gmail.com`)
- **Branch**: `feature/sruthi-single-image`
- **Target Base Model**: `google/paligemma-3b-pt-224`
- **Status**: `[PHASE 2 SMOKE TEST VERIFIED — RUNNING FULL TRAINING]`

> **Notice**: This notebook runs exclusively as an external GPU compute worker. The final SatQuery application runtime does not depend on Colab.

### Step 1: GPU Compute Environment & Hardware Diagnostics

In [1]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:    {torch.version.cuda}")

Tue Aug 25 13:41:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Secure Hugging Face Authentication

In [2]:
import os
import huggingface_hub

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    hf_token = getpass.getpass('Enter Hugging Face Access Token (Read role): ')

huggingface_hub.login(token=hf_token)
print("Hugging Face authentication completed securely.")

Hugging Face authentication completed securely.


### Step 3: Fast Git Repository Checkout & Dependency Setup

In [3]:
import os, sys
if not os.path.exists('/content/SatQuery'):
    !git clone https://github.com/Lalith2007/SatQuery.git /content/SatQuery
%cd /content/SatQuery
!git fetch origin
!git checkout feature/sruthi-single-image
!git reset --hard origin/feature/sruthi-single-image

# Fix Colab torchao conflict and install dependencies
!pip uninstall -y torchao
!pip install tqdm fastapi pydantic-settings tifffile pytest-asyncio peft
!pip install -e . --no-deps

if '/content/SatQuery' not in sys.path:
    sys.path.insert(0, '/content/SatQuery')

print("✓ Environment and repository ready!")

/content/SatQuery
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 7), reused 8 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.46 KiB | 299.00 KiB/s, done.
From https://github.com/Lalith2007/SatQuery
   c4bf1de..4488d81  feature/sruthi-single-image -> origin/feature/sruthi-single-image
M	specialists/single_image/weights/satquery_paligemma_lora/adapter_config.json
M	specialists/single_image/weights/satquery_paligemma_lora/adapter_model.safetensors
Already on 'feature/sruthi-single-image'
Your branch is behind 'origin/feature/sruthi-single-image' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 4488d81 feat(division-2): add tqdm progress tracking, live step loss, and sample count flags for full training
Obtaining file:///content/SatQuery
  Installing build dependencies ... done
  Checking if build backend suppor

### Step 4: Phase 1 — Real Model Load & Generation Verification

In [4]:
import torch, gc
from PIL import Image
from transformers import PaliGemmaForConditionalGeneration

model_id = "google/paligemma-3b-pt-224"

print(f"Loading {model_id} on GPU...")
try:
    from transformers import PaliGemmaProcessor
    processor = PaliGemmaProcessor.from_pretrained(model_id)
except Exception:
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained(model_id)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    device_map="cuda:0" if torch.cuda.is_available() else None,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"SUCCESS: Loaded {model_id} on {model.device}!")
print(f"Total Model Parameters: {total_params:,}")

# Test genuine model.generate() output
test_img = Image.new("RGB", (224, 224), color=(34, 139, 34))
inputs = processor(text="<image>answer en What is the dominant land cover?", images=test_img, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=32)
ans = processor.decode(output[0], skip_special_tokens=True)
print(f"Genuine Model Generation Output: '{ans}'")
print("PHASE 1 PASSED: REAL_MODEL_LOADED = True")

# Free Step 4 model memory so Step 5/6 has full GPU memory available
del model, processor
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed for Phase 2 & 3 training.")

Loading google/paligemma-3b-pt-224 on GPU...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

SUCCESS: Loaded google/paligemma-3b-pt-224 on cuda:0!
Total Model Parameters: 2,923,466,480
Genuine Model Generation Output: 'answer en What is the dominant land cover?
grass'
PHASE 1 PASSED: REAL_MODEL_LOADED = True
GPU memory freed for Phase 2 & 3 training.


### Step 5: Phase 2 — Real LoRA Gradient & Backprop Smoke Test (Verified Proof)

In [5]:
# Runs genuine forward pass, loss.backward(), non-zero gradient check, and optimizer.step() parameter delta
!python3 specialists/single_image/adaptation/train_lora.py --smoke-test --device cuda

2026-08-25 13:43:24 | INFO     | [satquery.train_lora] [train_lora.py:88] | ================================================================================
2026-08-25 13:43:24 | INFO     | [satquery.train_lora] [train_lora.py:89] | PHASE 2: REAL PEFT / LORA GRADIENT & BACKPROPAGATION SMOKE TEST ON [CUDA]
2026-08-25 13:43:32 | INFO     | [satquery.train_lora] [train_lora.py:105] | Loading base PaliGemma: google/paligemma-3b-pt-224 (revision: main)...
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files: 100% 3/3 [00:00<00:00, 948.65it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            0B                         
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 603/603 [00:47<00:00, 12.66it/s]
2026-08-25 13:44:25 | INFO     | [satquery.train_lora] [train_lora.py:139] | Total Parameters:     2,934,765,296
2026-08-25 13:44:25 | INFO     | [satquery.trai

### Step 6: Phase 3 — Real LoRA Domain Adaptation Training (3 Epochs with live loss & progress bar)

In [18]:
%cd /content/SatQuery
!git fetch origin
!git reset --hard origin/feature/sruthi-single-image


/content/SatQuery
HEAD is now at 2ff91e8 fix(division-2): update adapter_config.json to standard PEFT schema


In [19]:
!python3 specialists/single_image/adaptation/train_lora.py --epochs 3 --train-count 150 --val-count 30 --device cuda


2026-08-25 14:12:57 | INFO     | [satquery.train_lora] [train_lora.py:289] | ================================================================================
2026-08-25 14:12:57 | INFO     | [satquery.train_lora] [train_lora.py:290] | STARTING REAL LORA DOMAIN ADAPTATION TRAINING ON [CUDA]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files: 100% 3/3 [00:00<00:00, 831.76it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            0B                         
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 603/603 [00:47<00:00, 12.70it/s]
2026-08-25 14:13:56 | INFO     | [satquery.train_lora] [train_lora.py:334] | Total Params: 2,934,765,296 | Trainable: 11,298,816 (0.3850%)
2026-08-25 14:13:56 | INFO     | [satquery.train_lora] [train_lora.py:339] | Dataset Splits: Train=150, Val=30, Test=150
Epoch [1/3]: 100% 150/150 [01:04<00:00,  2.34sample/s, loss=2.43

### Step 7: Phase 4 & 5 — Real Model Evaluation & Synchronized CUDA Latency Benchmark

In [17]:
%cd /content/SatQuery
!git fetch origin
!git reset --hard origin/feature/sruthi-single-image


/content/SatQuery
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 1.01 KiB | 344.00 KiB/s, done.
From https://github.com/Lalith2007/SatQuery
   efb6c61..2ff91e8  feature/sruthi-single-image -> origin/feature/sruthi-single-image
HEAD is now at 2ff91e8 fix(division-2): update adapter_config.json to standard PEFT schema


In [20]:
!python3 specialists/single_image/evaluation/reproducibility.py


2026-08-25 14:17:24 | INFO     | [satquery.reproducibility] [reproducibility.py:364] | Executing Division 2 Real Adaptation Scientific Verification Audit...
2026-08-25 14:17:25 | INFO     | [satquery.single_image_model] [model.py:102] | Loading PaliGemma RS engine on device: 'cuda' (Base: google/paligemma-3b-pt-224)...
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files: 100% 3/3 [00:00<00:00, 992.58it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            0B                         
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 603/603 [00:47<00:00, 12.75it/s]
2026-08-25 14:18:23 | INFO     | [satquery.single_image_model] [model.py:129] | Loading SatQuery LoRA adapter from: specialists/single_image/weights/satquery_paligemma_lora
2026-08-25 14:18:24 | INFO     | [satquery.single_image_model] [model.py:134] | Successfully loaded REAL PaliGemma model 

### Step 8: Phase 6 — Export Reproducibility Manifest & Artifact Archive

In [21]:
!python3 specialists/single_image/colab/reproducibility_manifest.py
!tar -czvf satquery_division2_adapter_package.tar.gz specialists/single_image/weights/ specialists/single_image/evaluation/ specialists/single_image/colab/

from google.colab import files
files.download('/content/SatQuery/satquery_division2_adapter_package.tar.gz')


2026-08-25 14:25:19 | INFO     | [satquery.reproducibility_manifest] [reproducibility_manifest.py:73] | Generating SatQuery Division 2 Reproducibility Manifest...
2026-08-25 14:25:25 | INFO     | [satquery.reproducibility_manifest] [reproducibility_manifest.py:179] | Exported reproducibility manifest to: specialists/single_image/colab/reproducibility_manifest.json

SATQUERY REAL ADAPTATION REPRODUCIBILITY MANIFEST
Git Commit:   2ff91e8e
Branch:       feature/sruthi-single-image (sruthi-270 <rajamanurisruthi@gmail.com>)
Base Model:   google/paligemma-3b-pt-224
LoRA Adapter: SatQuery-PaliGemma-3B-RS-LoRA (SHA-256: 152075b5450b...)
Dataset Mix:  900 train, 150 test
Metrics:      VQA 43.5% ➔ 52.9% | Grounding mIoU 0.157 ➔ 0.265
specialists/single_image/weights/
specialists/single_image/weights/satquery_paligemma_lora/
specialists/single_image/weights/satquery_paligemma_lora/smoke_test_proof.json
specialists/single_image/weights/satquery_paligemma_lora/training_metrics.json
specialists/sing

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
import getpass
gh_token = getpass.getpass("Enter GitHub Access Token: ")
!git remote set-url origin https://{gh_token}@github.com/Lalith2007/SatQuery.git
!git push origin feature/sruthi-single-image


Enumerating objects: 32, done.
Counting objects: 100% (32/32), done.
Delta compression using up to 2 threads
Compressing objects: 100% (19/19), done.
Writing objects: 100% (19/19), 45.15 MiB | 3.81 MiB/s, done.
Total 19 (delta 8), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (8/8), completed with 8 local objects.
To https://github.com/Lalith2007/SatQuery.git
   2ff91e8..a4f0c4c  feature/sruthi-single-image -> feature/sruthi-single-image
